In [1]:
# Libraries
import numpy as np 
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform, transform_bounds # Reprojection
from rasterio.transform import Affine
np.set_printoptions(suppress = True) # Turn off scientific notation

In [ ]:
#### ssp2 2025: precipitation rate ########################################################################################

In [2]:
# Veiw raw data

ssp2_2025_prcp_rate = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate.tif'

with rasterio.open(ssp2_2025_prcp_rate, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float64', 'nodata': None, 'width': 237, 'height': 105, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.25, 0.0, 235.0,
       0.0, 0.25, 24.0), 'blockxsize': 237, 'blockysize': 4, 'tiled': False, 'interleave': 'band'}
Nodata: None
CRS: EPSG:4326
Resolution (0.25, 0.25)
Min: inf
Max: -inf
% nodata cells: 0.0


In [3]:
# Check nan cells
with rasterio.open(ssp2_2025_prcp_rate) as src:
    arr = src.read(1)
    print(np.isnan(arr).sum())

6867


In [4]:
# Update profile 
    # nodata = -10 
    # dtype = float32
    # compress = 'lzw'
    # tiled = True
    # blockxsize = 128
    # blockysize = 128

ssp2_2025_prcp_rate = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate.tif'
ssp2_2025_prcp_rate_profile_update = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_profile_update.tif'

with rasterio.open(ssp2_2025_prcp_rate) as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.float32,
                   nodata = -10,
                   compress = 'lzw',
                   tiled = True,
                   blockxsize = 128,
                   blockysize = 128)

    with rasterio.open(ssp2_2025_prcp_rate_profile_update, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.float64)
            
            # Replace nan cells to -10 (new nodata value)
            data[np.isnan(data)] = -10
            
            # Write out new raster
            dst.write(data.astype(rasterio.float32), 1, window = window)

In [5]:
# Verify profile update

ssp2_2025_prcp_rate_profile_update = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_profile_update.tif'

with rasterio.open(ssp2_2025_prcp_rate_profile_update, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 237, 'height': 105, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.25, 0.0, 235.0,
       0.0, 0.25, 24.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:4326
Resolution (0.25, 0.25)
Min: 0.14021961390972137
Max: 10.49191665649414
% nodata cells: 0.2759493670886076


In [6]:
# Reproject to epsg:5070, 30 m resolution (compression lzw) - 26 min

ssp2_2025_prcp_rate_profile_update = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_profile_update.tif'
ssp2_2025_prcp_rate_reprojected = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_reprojected.tif'

dst_crs = 'epsg:5070'  # Target CRS
res = 30 # Res in m

with rasterio.open(ssp2_2025_prcp_rate_profile_update) as src:
    # Calculate transform matrix for output
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src.crs, 
        dst_crs, 
        src.width, 
        src.height, 
        *src.bounds, # Unpacks bounds (left, bottom, right, top)
        resolution = res
    )

    # Set output properties
    dst_profile = src.profile.copy()
    dst_profile.update(
        crs = dst_crs,
        transform = dst_transform,
        width = dst_width,
        height = dst_height,
        nodata = -10,
        compress = 'lzw',
        tiled = True,
        blockxsize = 128,
        blockysize = 128,
        BIGTIFF = 'yes'
    )

    # Reproject each band
    with rasterio.open(ssp2_2025_prcp_rate_reprojected, 'w', **dst_profile) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source = rasterio.band(src, i),
                destination = rasterio.band(dst, i),
                src_transform = src.transform,
                src_crs = src.crs,
                dst_transform = dst_transform,
                dst_crs = dst_crs,
                resampling = Resampling.bilinear
            )

In [7]:
# Verify reprojection

ssp2_2025_prcp_rate_reprojected = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_reprojected.tif'

with rasterio.open(ssp2_2025_prcp_rate_reprojected, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 200994, 'height': 108747, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2950365.5094188303,
       0.0, -30.0, 3371941.7118588467), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CR

In [9]:
# Co-register with dist to gwt raster (inches convert) - 24 min

# Define co-register function - BILINEAR RESAMPLING and PROFILE UPDATE
def coregister_rasters(infile, match, outfile):
    """Reproject a file to match the shape and projection of existing raster. 
    
    Parameters
    ----------
    infile : (string) path to input file to reproject
    match : (string) path to raster with desired shape and projection 
    outfile : (string) path to output file tif
    """
    # Open input
    with rasterio.open(infile) as src:
        src_transform = src.transform
        
        # Open input to match
        with rasterio.open(match) as match:
            dst_crs = match.crs
            dst_transform = match.transform # Ensures resolutions of outfile and match will be exactly the same
            dst_width = match.width
            dst_height = match.height

        # Set properties for output
        dst_kwargs = src.meta.copy()
        dst_kwargs.update({'crs': dst_crs,
                           'transform': dst_transform,
                           'width': dst_width,
                           'height': dst_height,
                           'nodata': -10,
                           'compress': 'lzw',
                           'tiled': True,
                           'blockxsize': 128,
                           'blockysize': 128,
                           'BIGTIFF': 'yes'})
        print('Coregistered to shape:', dst_height, dst_width,'\n Affine', dst_transform)
        
        # Open output
        with rasterio.open(outfile, "w", **dst_kwargs) as dst:
            # Iterate through bands and write using reproject function
            for i in range(1, src.count + 1):
                reproject(
                    source = rasterio.band(src, i),
                    destination = rasterio.band(dst, i),
                    src_transform = src.transform,
                    src_crs = src.crs,
                    dst_transform = dst_transform,
                    dst_crs = dst_crs,
                    resampling = Resampling.bilinear)
                

# Apply coregister_rasters
ssp2_2025_prcp_rate_reprojected = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_reprojected.tif'
ref_raster = './data/SSURGO_raw/dist_GWT/gwt_inches.tif' # Match
ssp2_2025_prcp_rate_coregistered = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_coregistered.tif'

coregister_rasters(
    infile = ssp2_2025_prcp_rate_reprojected,
    match = ref_raster,
    outfile = ssp2_2025_prcp_rate_coregistered
)

Coregistered to shape: 96751 153996 
 Affine | 30.00, 0.00,-2356125.00|
| 0.00,-30.00, 3172575.00|
| 0.00, 0.00, 1.00|


In [10]:
# Verify co-register

ssp2_2025_prcp_rate_coregistered = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_coregistered.tif'

with rasterio.open(ssp2_2025_prcp_rate_coregistered, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [11]:
# Remove "excess" cells - started 138
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif' 
ssp2_2025_prcp_rate_coregistered = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_coregistered.tif'
ssp2_2025_prcp_rate_MATCH = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_MATCH.tif'

# Open hsg composite raster - want conus cells to match THIS raster
with rasterio.open(hsg_final_composite) as conus:

    # Nodata value for the conus raster
    conus_nodata = conus.nodata 
    # Profile conus raster
    profile = conus.profile.copy()
    profile.update(dtype = rasterio.float32,
                   tiled = True,
                   blockxsize = 128,
                   bloxkysize = 128,
                   compress = 'DEFLATE',
                   predictor = 3,
                   BIGTIFF = 'yes') 


    # Open raster - want to convert any cells containing data where conus contains NODATA to nodata
    with rasterio.open(ssp2_2025_prcp_rate_coregistered) as src:
        
        src_nodata = src.nodata # Nodata value 
        
        # Open output raster
        with rasterio.open(ssp2_2025_prcp_rate_MATCH, 'w', **profile) as dst:
        
            for ji, window in conus.block_windows(1):
            
                # hsg composite raster data
                conus_data = conus.read(1, window = window)
            
                # Land cover raster data
                src_data = src.read(1, window = window)
            
                # Identify cells where conus_data == nodata value
                remove_mask = (conus_data == conus_nodata)
            
                # Convert cells in src where conus is nodata to the nodata value
                src_data[remove_mask] = src_nodata
            
                # Write out
                dst.write(src_data, 1, window = window)

In [12]:
# Verify matching

ssp2_2025_prcp_rate_MATCH = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_MATCH.tif'

with rasterio.open(ssp2_2025_prcp_rate_MATCH, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Re

In [3]:
# Apply min-max scaling  ~30 min

ssp2_2025_prcp_rate_MATCH = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_MATCH.tif'
ssp2_2025_prcp_rate_standardized = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_standardized.tif'

with rasterio.open(ssp2_2025_prcp_rate_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    profile.update(BIGTIFF = 'yes',
                   compress = 'DEFLATE', 
                   predictor = 3)
    
    # Set initial global min and max
    global_min = np.inf # Highest possible number so anythign will automatically be less
    global_max = -np.inf # Lowest possible number so anything will automatically be greater
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
    # Sanity check: make sure min and max are reasonable values
    print(f'Global min: {global_min}')
    print(f'Global max: {global_max}')
    
    with rasterio.open(ssp2_2025_prcp_rate_standardized, mode = 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window, masked = True)
            
            # Apply min-max scaling
            rescaled = ((data - global_min) / (global_max - global_min) * 10)
            
            # Fill masked (nodata values) with the nodata value
            rescaled_filled = rescaled.filled(src.nodata)
            
            # Write out raster
            dst.write(rescaled_filled.astype(np.float32), 1, window = window)

Global min: 0.18370777368545532
Global max: 10.491494178771973


In [4]:
# Verify min-max scaling

ssp2_2025_prcp_rate_standardized = './data/projected_precip/precip_ssp2_2025/ssp2_2025_prcp_rate_standardized.tif'

with rasterio.open(ssp2_2025_prcp_rate_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Re

In [ ]:
#### ssp2 2050: precipitation rate ########################################################################################

In [5]:
# Veiw raw data

ssp2_2050_prcp_rate = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate.tif'

with rasterio.open(ssp2_2050_prcp_rate, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float64', 'nodata': None, 'width': 237, 'height': 105, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.25, 0.0, 235.0,
       0.0, 0.25, 24.0), 'blockxsize': 237, 'blockysize': 4, 'tiled': False, 'interleave': 'band'}
Nodata: None
CRS: EPSG:4326
Resolution (0.25, 0.25)
Min: inf
Max: -inf
% nodata cells: 0.0


In [6]:
# Check nan cells
with rasterio.open(ssp2_2050_prcp_rate) as src:
    arr = src.read(1)
    print(np.isnan(arr).sum())

6867


In [7]:
# Update profile 
    # nodata = -10 
    # dtype = float32
    # compress = 'lzw'
    # tiled = True
    # blockxsize = 128
    # blockysize = 128

ssp2_2050_prcp_rate = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate.tif'
ssp2_2050_prcp_rate_profile_update = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_update_profile.tif'

with rasterio.open(ssp2_2050_prcp_rate) as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.float32,
                   nodata = -10,
                   compress = 'lzw',
                   tiled = True,
                   blockxsize = 128,
                   blockysize = 128)

    with rasterio.open(ssp2_2050_prcp_rate_profile_update, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.float64)
            
            # Replace nan cells to -10 (new nodata value)
            data[np.isnan(data)] = -10
            
            # Write out new raster
            dst.write(data.astype(rasterio.float32), 1, window = window)

In [8]:
# Verify profile update

ssp2_2050_prcp_rate_profile_update = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_update_profile.tif'

with rasterio.open(ssp2_2050_prcp_rate_profile_update, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 237, 'height': 105, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.25, 0.0, 235.0,
       0.0, 0.25, 24.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:4326
Resolution (0.25, 0.25)
Min: 0.1097002848982811
Max: 11.155237197875977
% nodata cells: 0.2759493670886076


In [9]:
# Reproject to epsg:5070, 30 m resolution (compression lzw) - 26 min

ssp2_2050_prcp_rate_profile_update = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_update_profile.tif'
ssp2_2050_prcp_rate_reprojected = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_reprojected.tif'

dst_crs = 'epsg:5070'  # Target CRS
res = 30 # Res in m

with rasterio.open(ssp2_2050_prcp_rate_profile_update) as src:
    # Calculate transform matrix for output
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src.crs, 
        dst_crs, 
        src.width, 
        src.height, 
        *src.bounds, # Unpacks bounds (left, bottom, right, top)
        resolution = res
    )

    # Set output properties
    dst_profile = src.profile.copy()
    dst_profile.update(
        crs = dst_crs,
        transform = dst_transform,
        width = dst_width,
        height = dst_height,
        nodata = -10,
        compress = 'lzw',
        tiled = True,
        blockxsize = 128,
        blockysize = 128,
        BIGTIFF = 'yes'
    )

    # Reproject each band
    with rasterio.open(ssp2_2050_prcp_rate_reprojected, 'w', **dst_profile) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source = rasterio.band(src, i),
                destination = rasterio.band(dst, i),
                src_transform = src.transform,
                src_crs = src.crs,
                dst_transform = dst_transform,
                dst_crs = dst_crs,
                resampling = Resampling.bilinear
            )

In [10]:
# Verify reprojection

ssp2_2050_prcp_rate_reprojected = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_reprojected.tif'

with rasterio.open(ssp2_2050_prcp_rate_reprojected, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 200994, 'height': 108747, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2950365.5094188303,
       0.0, -30.0, 3371941.7118588467), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CR

In [11]:
# Co-register with dist to gwt raster (inches convert) - 24 min

# Define co-register function - BILINEAR RESAMPLING and PROFILE UPDATE
def coregister_rasters(infile, match, outfile):
    """Reproject a file to match the shape and projection of existing raster. 
    
    Parameters
    ----------
    infile : (string) path to input file to reproject
    match : (string) path to raster with desired shape and projection 
    outfile : (string) path to output file tif
    """
    # Open input
    with rasterio.open(infile) as src:
        src_transform = src.transform
        
        # Open input to match
        with rasterio.open(match) as match:
            dst_crs = match.crs
            dst_transform = match.transform # Ensures resolutions of outfile and match will be exactly the same
            dst_width = match.width
            dst_height = match.height

        # Set properties for output
        dst_kwargs = src.meta.copy()
        dst_kwargs.update({'crs': dst_crs,
                           'transform': dst_transform,
                           'width': dst_width,
                           'height': dst_height,
                           'nodata': -10,
                           'compress': 'lzw',
                           'tiled': True,
                           'blockxsize': 128,
                           'blockysize': 128,
                           'BIGTIFF': 'yes'})
        print('Coregistered to shape:', dst_height, dst_width,'\n Affine', dst_transform)
        
        # Open output
        with rasterio.open(outfile, "w", **dst_kwargs) as dst:
            # Iterate through bands and write using reproject function
            for i in range(1, src.count + 1):
                reproject(
                    source = rasterio.band(src, i),
                    destination = rasterio.band(dst, i),
                    src_transform = src.transform,
                    src_crs = src.crs,
                    dst_transform = dst_transform,
                    dst_crs = dst_crs,
                    resampling = Resampling.bilinear)
                

# Apply coregister_rasters
ssp2_2050_prcp_rate_reprojected = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_reprojected.tif'
ref_raster = './data/SSURGO_raw/dist_GWT/gwt_inches.tif' # Match
ssp2_2050_prcp_rate_coregistered = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_coregistered.tif'

coregister_rasters(
    infile = ssp2_2050_prcp_rate_reprojected,
    match = ref_raster,
    outfile = ssp2_2050_prcp_rate_coregistered
)

Coregistered to shape: 96751 153996 
 Affine | 30.00, 0.00,-2356125.00|
| 0.00,-30.00, 3172575.00|
| 0.00, 0.00, 1.00|


In [12]:
# Verify co-register

ssp2_2050_prcp_rate_coregistered = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_coregistered.tif'

with rasterio.open(ssp2_2050_prcp_rate_coregistered, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [13]:
# Remove "excess" cells - started 138
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif' 
ssp2_2050_prcp_rate_coregistered = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_coregistered.tif'
ssp2_2050_prcp_rate_MATCH = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_MATCH.tif'

# Open hsg composite raster - want conus cells to match THIS raster
with rasterio.open(hsg_final_composite) as conus:

    # Nodata value for the conus raster
    conus_nodata = conus.nodata 
    # Profile conus raster
    profile = conus.profile.copy()
    profile.update(dtype = rasterio.float32,
                   tiled = True,
                   blockxsize = 128,
                   bloxkysize = 128,
                   compress = 'DEFLATE',
                   predictor = 3,
                   BIGTIFF = 'yes') 


    # Open raster - want to convert any cells containing data where conus contains NODATA to nodata
    with rasterio.open(ssp2_2050_prcp_rate_coregistered) as src:
        
        src_nodata = src.nodata # Nodata value 
        
        # Open output raster
        with rasterio.open(ssp2_2050_prcp_rate_MATCH, 'w', **profile) as dst:
        
            for ji, window in conus.block_windows(1):
            
                # hsg composite raster data
                conus_data = conus.read(1, window = window)
            
                # Land cover raster data
                src_data = src.read(1, window = window)
            
                # Identify cells where conus_data == nodata value
                remove_mask = (conus_data == conus_nodata)
            
                # Convert cells in src where conus is nodata to the nodata value
                src_data[remove_mask] = src_nodata
            
                # Write out
                dst.write(src_data, 1, window = window)

In [14]:
# Verify matching

ssp2_2050_prcp_rate_MATCH = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_MATCH.tif'

with rasterio.open(ssp2_2050_prcp_rate_MATCH, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Re

In [15]:
# Apply min-max scaling  ~30 min

ssp2_2050_prcp_rate_MATCH = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_MATCH.tif'
ssp2_2050_prcp_rate_standardized = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_standardized.tif'

with rasterio.open(ssp2_2050_prcp_rate_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    profile.update(BIGTIFF = 'yes',
                   compress = 'DEFLATE', 
                   predictor = 3)
    
    # Set initial global min and max
    global_min = np.inf # Highest possible number so anythign will automatically be less
    global_max = -np.inf # Lowest possible number so anything will automatically be greater
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
    # Sanity check: make sure min and max are reasonable values
    print(f'Global min: {global_min}')
    print(f'Global max: {global_max}')
    
    with rasterio.open(ssp2_2050_prcp_rate_standardized, mode = 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window, masked = True)
            
            # Apply min-max scaling
            rescaled = ((data - global_min) / (global_max - global_min) * 10)
            
            # Fill masked (nodata values) with the nodata value
            rescaled_filled = rescaled.filled(src.nodata)
            
            # Write out raster
            dst.write(rescaled_filled.astype(np.float32), 1, window = window)

Global min: 0.1254815012216568
Global max: 11.154765129089355


In [16]:
# Verify min-max scaling

ssp2_2050_prcp_rate_standardized = './data/projected_precip/precip_ssp2_2050/ssp2_2050_prcp_rate_standardized.tif'

with rasterio.open(ssp2_2050_prcp_rate_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Re

In [ ]:
#### ssp5 2050: precipitation rate ########################################################################################

In [17]:
# Veiw raw data

ssp5_2050_prcp_rate = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate.tif'

with rasterio.open(ssp5_2050_prcp_rate, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float64', 'nodata': None, 'width': 237, 'height': 105, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.25, 0.0, 235.0,
       0.0, 0.25, 24.0), 'blockxsize': 237, 'blockysize': 4, 'tiled': False, 'interleave': 'band'}
Nodata: None
CRS: EPSG:4326
Resolution (0.25, 0.25)
Min: inf
Max: -inf
% nodata cells: 0.0


In [18]:
# Check nan cells
with rasterio.open(ssp5_2050_prcp_rate) as src:
    arr = src.read(1)
    print(np.isnan(arr).sum())

6867


In [19]:
# Update profile 
    # nodata = -10 
    # dtype = float32
    # compress = 'lzw'
    # tiled = True
    # blockxsize = 128
    # blockysize = 128

ssp5_2050_prcp_rate = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate.tif'
ssp5_2050_prcp_rate_profile_update = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_profile_update.tif'

with rasterio.open(ssp5_2050_prcp_rate) as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.float32,
                   nodata = -10,
                   compress = 'lzw',
                   tiled = True,
                   blockxsize = 128,
                   blockysize = 128)

    with rasterio.open(ssp5_2050_prcp_rate_profile_update, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.float64)
            
            # Replace nan cells to -10 (new nodata value)
            data[np.isnan(data)] = -10
            
            # Write out new raster
            dst.write(data.astype(rasterio.float32), 1, window = window)

In [20]:
# Verify profile update

ssp5_2050_prcp_rate_profile_update = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_profile_update.tif'

with rasterio.open(ssp5_2050_prcp_rate_profile_update, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 237, 'height': 105, 'count': 1, 'crs': CRS.from_wkt('GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]'), 'transform': Affine(0.25, 0.0, 235.0,
       0.0, 0.25, 24.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:4326
Resolution (0.25, 0.25)
Min: 0.10358243435621262
Max: 10.583086967468262
% nodata cells: 0.2759493670886076


In [2]:
# Reproject to epsg:5070, 30 m resolution (compression lzw) - 26 min

ssp5_2050_prcp_rate_profile_update = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_profile_update.tif'
ssp5_2050_prcp_rate_reprojected = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_reprojected.tif'

dst_crs = 'epsg:5070'  # Target CRS
res = 30 # Res in m

with rasterio.open(ssp5_2050_prcp_rate_profile_update) as src:
    # Calculate transform matrix for output
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src.crs, 
        dst_crs, 
        src.width, 
        src.height, 
        *src.bounds, # Unpacks bounds (left, bottom, right, top)
        resolution = res
    )

    # Set output properties
    dst_profile = src.profile.copy()
    dst_profile.update(
        crs = dst_crs,
        transform = dst_transform,
        width = dst_width,
        height = dst_height,
        nodata = -10,
        compress = 'lzw',
        tiled = True,
        blockxsize = 128,
        blockysize = 128,
        BIGTIFF = 'yes'
    )

    # Reproject each band
    with rasterio.open(ssp5_2050_prcp_rate_reprojected, 'w', **dst_profile) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source = rasterio.band(src, i),
                destination = rasterio.band(dst, i),
                src_transform = src.transform,
                src_crs = src.crs,
                dst_transform = dst_transform,
                dst_crs = dst_crs,
                resampling = Resampling.bilinear
            )

In [3]:
# Verify reprojection

ssp5_2050_prcp_rate_reprojected = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_reprojected.tif'

with rasterio.open(ssp5_2050_prcp_rate_reprojected, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 200994, 'height': 108747, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2950365.5094188303,
       0.0, -30.0, 3371941.7118588467), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CR

In [4]:
# Co-register with dist to gwt raster (inches convert) - 24 min

# Define co-register function - BILINEAR RESAMPLING and PROFILE UPDATE
def coregister_rasters(infile, match, outfile):
    """Reproject a file to match the shape and projection of existing raster. 
    
    Parameters
    ----------
    infile : (string) path to input file to reproject
    match : (string) path to raster with desired shape and projection 
    outfile : (string) path to output file tif
    """
    # Open input
    with rasterio.open(infile) as src:
        src_transform = src.transform
        
        # Open input to match
        with rasterio.open(match) as match:
            dst_crs = match.crs
            dst_transform = match.transform # Ensures resolutions of outfile and match will be exactly the same
            dst_width = match.width
            dst_height = match.height

        # Set properties for output
        dst_kwargs = src.meta.copy()
        dst_kwargs.update({'crs': dst_crs,
                           'transform': dst_transform,
                           'width': dst_width,
                           'height': dst_height,
                           'nodata': -10,
                           'compress': 'lzw',
                           'tiled': True,
                           'blockxsize': 128,
                           'blockysize': 128,
                           'BIGTIFF': 'yes'})
        print('Coregistered to shape:', dst_height, dst_width,'\n Affine', dst_transform)
        
        # Open output
        with rasterio.open(outfile, "w", **dst_kwargs) as dst:
            # Iterate through bands and write using reproject function
            for i in range(1, src.count + 1):
                reproject(
                    source = rasterio.band(src, i),
                    destination = rasterio.band(dst, i),
                    src_transform = src.transform,
                    src_crs = src.crs,
                    dst_transform = dst_transform,
                    dst_crs = dst_crs,
                    resampling = Resampling.bilinear)
                

# Apply coregister_rasters
ssp5_2050_prcp_rate_reprojected = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_reprojected.tif'
ref_raster = './data/SSURGO_raw/dist_GWT/gwt_inches.tif' # Match
ssp5_2050_prcp_rate_coregistered = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_coregistered.tif'

coregister_rasters(
    infile = ssp5_2050_prcp_rate_reprojected,
    match = ref_raster,
    outfile = ssp5_2050_prcp_rate_coregistered
)

Coregistered to shape: 96751 153996 
 Affine | 30.00, 0.00,-2356125.00|
| 0.00,-30.00, 3172575.00|
| 0.00, 0.00, 1.00|


In [5]:
# Verify co-register

ssp5_2050_prcp_rate_coregistered = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_coregistered.tif'

with rasterio.open(ssp5_2050_prcp_rate_coregistered, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [6]:
# Remove "excess" cells - started 138
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif' 
ssp5_2050_prcp_rate_coregistered = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_coregistered.tif'
ssp5_2050_prcp_rate_MATCH = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_MATCH.tif'

# Open hsg composite raster - want conus cells to match THIS raster
with rasterio.open(hsg_final_composite) as conus:

    # Nodata value for the conus raster
    conus_nodata = conus.nodata 
    # Profile conus raster
    profile = conus.profile.copy()
    profile.update(dtype = rasterio.float32,
                   tiled = True,
                   blockxsize = 128,
                   bloxkysize = 128,
                   compress = 'DEFLATE',
                   predictor = 3,
                   BIGTIFF = 'yes') 


    # Open raster - want to convert any cells containing data where conus contains NODATA to nodata
    with rasterio.open(ssp5_2050_prcp_rate_coregistered) as src:
        
        src_nodata = src.nodata # Nodata value 
        
        # Open output raster
        with rasterio.open(ssp5_2050_prcp_rate_MATCH, 'w', **profile) as dst:
        
            for ji, window in conus.block_windows(1):
            
                # hsg composite raster data
                conus_data = conus.read(1, window = window)
            
                # Land cover raster data
                src_data = src.read(1, window = window)
            
                # Identify cells where conus_data == nodata value
                remove_mask = (conus_data == conus_nodata)
            
                # Convert cells in src where conus is nodata to the nodata value
                src_data[remove_mask] = src_nodata
            
                # Write out
                dst.write(src_data, 1, window = window)

In [7]:
# Verify matching

ssp5_2050_prcp_rate_MATCH = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_MATCH.tif'

with rasterio.open(ssp5_2050_prcp_rate_MATCH, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Re

In [8]:
# Apply min-max scaling  ~30 min

ssp5_2050_prcp_rate_MATCH = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_MATCH.tif'
ssp5_2050_prcp_rate_standardized = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_standardized.tif'

with rasterio.open(ssp5_2050_prcp_rate_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    profile.update(BIGTIFF = 'yes',
                   compress = 'DEFLATE', 
                   predictor = 3)
    
    # Set initial global min and max
    global_min = np.inf # Highest possible number so anythign will automatically be less
    global_max = -np.inf # Lowest possible number so anything will automatically be greater
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
    # Sanity check: make sure min and max are reasonable values
    print(f'Global min: {global_min}')
    print(f'Global max: {global_max}')
    
    with rasterio.open(ssp5_2050_prcp_rate_standardized, mode = 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window, masked = True)
            
            # Apply min-max scaling
            rescaled = ((data - global_min) / (global_max - global_min) * 10)
            
            # Fill masked (nodata values) with the nodata value
            rescaled_filled = rescaled.filled(src.nodata)
            
            # Write out raster
            dst.write(rescaled_filled.astype(np.float32), 1, window = window)

Global min: 0.10358978807926178
Global max: 10.582666397094727


In [9]:
# Verify min-max scaling

ssp5_2050_prcp_rate_standardized = './data/projected_precip/precip_ssp5_2050/ssp5_2050_prcp_rate_standardized.tif'

with rasterio.open(ssp5_2050_prcp_rate_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Re